## Figure 11b — EP100-window sensitivity

Plot action: render the stored Pearson-correlation scan for the January and
February hindcasts, hatch windows that overlap the earliest member ozone
minimum, and mark only the selected day-21/20-day window with one diamond per
panel.

Inputs: `relationships/figure11.nc`, containing Pearson r, p, n, exact start
and end DOY, and the precomputed overlap-safety mask for skips 1–60 and window
lengths 7–45 days. No optimum is selected by this plotting block.

Outputs: `figure11b.png` and PDF.

**Typography.** The plotting cell applies the shared publication-scale Paper 1 style immediately before saving: enlarged titles, axis labels, ticks, legends, and colour-bar text at the final manuscript canvas size.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    import sys as _sys
    style_directory = str(REPOSITORY_ROOT / "figures")
    if style_directory not in _sys.path:
        _sys.path.insert(0, style_directory)
    from paper_style import apply_paper_style
    apply_paper_style(figure, stem)
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
product = load_dataset(
    "relationships/figure11.nc",
    ("r", "p", "n", "safe", "window_start_doy", "window_end_doy"),
)
expected_dims = ("case", "length", "skip")
if product["r"].dims != expected_dims:
    raise ValueError(f"Unexpected Figure 11 dimensions {product['r'].dims}")
if str(product.attrs.get("selected_marker", "")) != "skip=21,length=20; one diamond per panel":
    raise ValueError("Figure 11 product lacks the selected 21/20 marker contract")
case_values = list(product["case"].values)
case_labels = [text_value(value) for value in case_values]
lengths = np.asarray(product["length"].values, dtype=float)
skips = np.asarray(product["skip"].values, dtype=float)
figure, axes = plt.subplots(
    1, len(case_values), figsize=(15.2, 5.9), sharex=True, sharey=True,
    constrained_layout=True,
)
axes = np.atleast_1d(axes)
mappable = None
labels = {"0008-01": "January initialization", "0008-02": "February initialization"}
for axis, case, case_label in zip(axes, case_values, case_labels):
    stored_r = np.asarray(product["r"].sel(case=case).values, dtype=float)
    safe = np.asarray(product["safe"].sel(case=case).values, dtype=bool)
    mappable = axis.pcolormesh(
        skips, lengths, stored_r, cmap="RdBu_r", vmin=-1, vmax=1, shading="auto",
    )
    unsafe = np.where(np.isfinite(stored_r) & (~safe), 1.0, np.nan)
    axis.contourf(
        skips, lengths, unsafe, levels=[0.5, 1.5], colors=["none"],
        hatches=["///"], zorder=3,
    )
    axis.scatter(
        [21], [20], marker="D", s=58, facecolor="white", edgecolor="black",
        zorder=6, label="days 21–40 (20 days)",
    )
    axis.set_title(labels.get(case_label, case_label), fontweight="bold")
    axis.set_xlabel("Window start day counted from initialization day")
    axis.grid(True, color="0.75", ls=":", lw=0.5, alpha=0.7)
axes[0].set_ylabel("Continuous averaging length (days)")
handles, legend_labels = axes[1].get_legend_handles_labels()
figure.legend(handles, legend_labels, loc="upper center", frameon=True,
              bbox_to_anchor=(0.5, 1.03))
colorbar = figure.colorbar(mappable, ax=axes, pad=0.02, fraction=0.035)
colorbar.set_label("Pearson correlation: EP100 window mean versus O$_3$ minimum")
figure.suptitle("EP100-window sensitivity before ozone-minimum overlap", fontsize=14)
save_figure(figure, "figure11b")
